In [3]:
import sys
import os

# Get the root directory (two levels up from this notebook)
root_dir = os.path.abspath(os.path.join(os.getcwd(), "../"))
sys.path.append(root_dir)

# Now import
from Python_Helper.wiki_extract import extract_wikipedia_page_optimized

In [ ]:
"""
Wikipedia Data Extraction Pipeline
A robust, production-ready pipeline for extracting and managing Wikipedia data
with advanced features including pause/resume, deduplication, and comprehensive logging.
"""
import sys
import os

# Get the root directory (two levels up from this notebook)
root_dir = os.path.abspath(os.path.join(os.getcwd(), "../../../"))
sys.path.append(root_dir)


# Now import
from Python_Helper.wiki_extract  import extract_wikipedia_page_optimized

import json
import os
import time
import logging
import asyncio
import hashlib
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set, Optional, Any, Tuple
from dataclasses import dataclass, asdict, field
from datetime import datetime
from enum import Enum
import signal
import sys
from collections import deque
import threading
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
import traceback

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('pipeline.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class PipelineState(Enum):
    """Pipeline execution states"""
    IDLE = "idle"
    RUNNING = "running"
    PAUSED = "paused"
    STOPPED = "stopped"
    ERROR = "error"
    COMPLETED = "completed"

@dataclass
class WikiEntity:
    """Data class for Wikipedia entities"""
    qid: str
    title: str
    type: str
    shortDesc: Optional[str] = None
    redirect_title: Optional[str] = None
    extraction_time: Optional[str] = None
    depth: int = 0
    parent_qid: Optional[str] = None
    
    def get_effective_title(self) -> str:
        """Get the effective title (redirect if exists, otherwise original)"""
        return self.redirect_title if self.redirect_title else self.title
    
    def to_dict(self) -> Dict:
        """Convert to dictionary"""
        return asdict(self)

@dataclass
class PipelineConfig:
    """Configuration for the pipeline"""
    base_dir: str = "wikipedia_data"
    max_depth: int = 3
    batch_size: int = 10
    max_workers: int = 5
    pause_between_requests: float = 1.0
    auto_save_interval: int = 10  # Save state every N extractions
    enable_deduplication: bool = True
    interactive_mode: bool = True
    retry_attempts: int = 3
    retry_delay: float = 2.0

class PipelineStatistics:
    """Track pipeline statistics"""
    def __init__(self):
        self.total_extracted = 0
        self.total_errors = 0
        self.total_duplicates = 0
        self.extraction_times = []
        self.start_time = None
        self.end_time = None
        
    def add_extraction_time(self, duration: float):
        self.extraction_times.append(duration)
        
    def get_average_extraction_time(self) -> float:
        if not self.extraction_times:
            return 0
        return sum(self.extraction_times) / len(self.extraction_times)
    
    def get_summary(self) -> Dict:
        runtime = 0
        if self.start_time:
            end = self.end_time or datetime.now()
            runtime = (end - self.start_time).total_seconds()
            
        return {
            "total_extracted": self.total_extracted,
            "total_errors": self.total_errors,
            "total_duplicates": self.total_duplicates,
            "average_extraction_time": self.get_average_extraction_time(),
            "total_runtime_seconds": runtime
        }

class WikipediaExtractionPipeline:
    """
    State-of-the-art Wikipedia extraction pipeline with advanced features
    """
    
    def __init__(self, config: Optional[PipelineConfig] = None):
        self.config = config or PipelineConfig()
        self.state = PipelineState.IDLE
        self.statistics = PipelineStatistics()
        
        # Data structures
        self.queue: deque = deque()
        self.processed_qids: Set[str] = set()
        self.error_qids: Dict[str, int] = {}  # QID -> retry count
        self.type_mapping: Dict[str, Set[str]] = {}  # type -> set of QIDs
        
        # Setup directories
        self._setup_directories()
        
        # Load previous state if exists
        self._load_state()
        
        # Setup signal handlers for graceful shutdown
        signal.signal(signal.SIGINT, self._signal_handler)
        signal.signal(signal.SIGTERM, self._signal_handler)
        
        # Thread lock for thread-safe operations
        self.lock = threading.Lock()
        
        logger.info("Pipeline initialized successfully")
    
    def _setup_directories(self):
        """Create necessary directory structure"""
        self.base_path = Path(self.config.base_dir)
        self.base_path.mkdir(exist_ok=True)
        
        # Create subdirectories
        self.state_dir = self.base_path.parent / "pipeline_state"
        self.state_dir.mkdir(exist_ok=True)
        
        self.logs_dir = self.base_path.parent / "logs"
        self.logs_dir.mkdir(exist_ok=True)
        
        logger.info(f"Directory structure created at {self.base_path}")
    
    def _signal_handler(self, signum, frame):
        """Handle interrupt signals gracefully"""
        logger.warning(f"Received signal {signum}. Initiating graceful shutdown...")
        self.pause()
        self._save_state()
        logger.info("Pipeline paused and state saved. You can resume later.")
        sys.exit(0)
    
    def _load_state(self):
        """Load previous pipeline state if exists"""
        state_file = self.state_dir / "pipeline_state.pkl"
        if state_file.exists():
            try:
                with open(state_file, 'rb') as f:
                    state_data = pickle.load(f)
                    self.queue = state_data.get('queue', deque())
                    self.processed_qids = state_data.get('processed_qids', set())
                    self.error_qids = state_data.get('error_qids', {})
                    self.type_mapping = state_data.get('type_mapping', {})
                    self.statistics = state_data.get('statistics', PipelineStatistics())
                logger.info(f"Loaded previous state: {len(self.processed_qids)} processed, {len(self.queue)} in queue")
            except Exception as e:
                logger.error(f"Failed to load state: {e}")
    
    def _save_state(self):
        """Save current pipeline state"""
        state_file = self.state_dir / "pipeline_state.pkl"
        try:
            state_data = {
                'queue': self.queue,
                'processed_qids': self.processed_qids,
                'error_qids': self.error_qids,
                'type_mapping': self.type_mapping,
                'statistics': self.statistics,
                'timestamp': datetime.now().isoformat()
            }
            with open(state_file, 'wb') as f:
                pickle.dump(state_data, f)
            logger.debug("Pipeline state saved")
        except Exception as e:
            logger.error(f"Failed to save state: {e}")
    
    def _extract_wikipedia_page_mock(self, page_title: str) -> Dict:
        """
        Mock function to simulate extract_wikipedia_page_optimized
        Replace this with actual implementation
        """
        # Simulate extraction delay
        time.sleep(0.5)
        
        # Mock data structure similar to your example
        return {
            "title": page_title,
            "qid": f"Q{hash(page_title) % 1000000}",
            "type": "entity",
            "links": {
                "internal_links": [
                    {
                        "title": f"Link_{i}_from_{page_title}",
                        "qid": f"Q{hash(f'{page_title}_{i}') % 1000000}",
                        "type": ["person", "place", "event", "concept"][i % 4],
                        "shortDesc": f"Description for link {i}",
                        "redirectTitle": None if i % 3 else f"Redirect_{i}"
                    }
                    for i in range(3)  # Generate 3 mock links
                ]
            },
            "metadata": {
                "page_id": str(hash(page_title) % 10000000),
                "last_modified": datetime.now().isoformat()
            }
        }
    
    async def extract_entity(self, entity: WikiEntity) -> Optional[Dict]:
        """
        Extract data for a single entity with error handling and retries
        """
        title = entity.get_effective_title()
        
        for attempt in range(self.config.retry_attempts):
            try:
                start_time = time.time()
                
                # Call the extraction function (replace with actual)
                data = await extract_wikipedia_page_optimized(title)
                # For testing, use mock function
                # data = self._extract_wikipedia_page_mock(title)
                
                extraction_time = time.time() - start_time
                self.statistics.add_extraction_time(extraction_time)
                
                logger.info(f"Successfully extracted: {title} (QID: {entity.qid}) in {extraction_time:.2f}s")
                return data
                
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed for {title}: {e}")
                if attempt < self.config.retry_attempts - 1:
                    time.sleep(self.config.retry_delay * (attempt + 1))
                else:
                    logger.error(f"Failed to extract {title} after {self.config.retry_attempts} attempts")
                    with self.lock:
                        self.error_qids[entity.qid] = self.config.retry_attempts
                    self.statistics.total_errors += 1
                    return None
    
    def save_entity_data(self, data: Dict, entity: WikiEntity):
        """
        Save entity data to appropriate folder structure
        """
        try:
            # Determine type and create directory
            entity_type = str(data.get('type') or 'unknown').replace(' ', '_').replace('/', '_')            
            type_dir = self.base_path / entity_type
            type_dir.mkdir(exist_ok=True)
            
            # Save JSON file named with QID
            qid = data.get('qid', entity.qid)
            file_path = type_dir / f"{qid}.json"
            
            # Add extraction metadata
            data['extraction_metadata'] = {
                'timestamp': datetime.now().isoformat(),
                'depth': entity.depth,
                'parent_qid': entity.parent_qid,
                'pipeline_version': '2.0'
            }
            
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            
            # Update type mapping
            with self.lock:
                if entity_type not in self.type_mapping:
                    self.type_mapping[entity_type] = set()
                self.type_mapping[entity_type].add(qid)
            
            # Update type-specific Excel
            self._update_type_excel(entity_type, data, entity)
            
            logger.debug(f"Saved {qid} to {file_path}")
            
        except Exception as e:
            logger.error(f"Failed to save entity data: {e}")
            raise
    
    def _update_type_excel(self, entity_type: str, data: Dict, entity: WikiEntity):
        """
        Update Excel file for specific entity type
        """
        try:
            type_dir = self.base_path / entity_type
            excel_path = type_dir / f"{entity_type}_summary.xlsx"
            
            # Prepare row data
            row_data = {
                'qid': data.get('qid'),
                'title': data.get('title'),
                'type': entity_type,
                'shortDesc': entity.shortDesc,
                'redirect_title': entity.redirect_title,
                'extraction_time': datetime.now().isoformat(),
                'depth': entity.depth,
                'parent_qid': entity.parent_qid,
                'num_links': len(data.get('links', {}).get('internal_links', [])),
                'num_tables': len(data.get('tables', [])),
                'num_images': len(data.get('images', []))
            }
            
            # Load or create DataFrame
            if excel_path.exists():
                df = pd.read_excel(excel_path)
                # Check if QID already exists
                if data.get('qid') not in df['qid'].values:
                    df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)
                else:
                    # Update existing row
                    idx = df[df['qid'] == data.get('qid')].index[0]
                    for key, value in row_data.items():
                        df.at[idx, key] = value
            else:
                df = pd.DataFrame([row_data])
            
            # Save Excel
            df.to_excel(excel_path, index=False)
            
        except Exception as e:
            logger.error(f"Failed to update type Excel: {e}")
    
    def update_master_excel(self):
        """
        Update master Excel file with summary of all entities
        """
        try:
            master_path = self.base_path / "master_summary.xlsx"
            
            rows = []
            for entity_type, qids in self.type_mapping.items():
                type_dir = self.base_path / entity_type
                for qid in qids:
                    file_path = type_dir / f"{qid}.json"
                    if file_path.exists():
                        with open(file_path, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                            
                        rows.append({
                            'qid': qid,
                            'title': data.get('title'),
                            'type': entity_type,
                            'shortDesc':data.get('content', {}).get('description'),
                            'num_infobox_fields': len(data.get('infobox', {})),
                            'num_links': len(data.get('links', {}).get('internal_links', [])),
                            'num_tables': len(data.get('tables', [])),
                            'num_images': len(data.get('images', [])),
                            'num_chunks': len(data.get('chunks', [])),
                            'page_length': data.get('metadata', {}).get('page_length', 0),
                            'last_modified': data.get('metadata', {}).get('last_modified'),
                            'extraction_date': data.get('extraction_metadata', {}).get('timestamp')
                        })
            
            if rows:
                df = pd.DataFrame(rows)
                df.to_excel(master_path, index=False)
                logger.info(f"Master Excel updated with {len(rows)} entries")
            
        except Exception as e:
            logger.error(f"Failed to update master Excel: {e}")
    
    def process_entity_links(self, data: Dict, parent_entity: WikiEntity) -> List[WikiEntity]:
        """
        Process internal links from extracted data and create new entities
        """
        new_entities = []
        
        if parent_entity.depth >= self.config.max_depth:
            logger.debug(f"Max depth {self.config.max_depth} reached for {parent_entity.title}")
            return new_entities
        
        internal_links = data.get('links', {}).get('internal_links', [])
        
        for link in internal_links:
            qid = link.get('qid')
            
            # Skip if already processed or in error list
            if self.config.enable_deduplication:
                if qid in self.processed_qids:
                    self.statistics.total_duplicates += 1
                    logger.debug(f"Skipping duplicate: {qid}")
                    continue
                
                if qid in self.error_qids and self.error_qids[qid] >= self.config.retry_attempts:
                    logger.debug(f"Skipping error entity: {qid}")
                    continue
            
            # Create new entity
            new_entity = WikiEntity(
                qid=qid,
                title=link.get('title'),
                type=link.get('type', 'unknown'),
                shortDesc=link.get('shortDesc'),
                redirect_title=link.get('redirectTitle'),
                depth=parent_entity.depth + 1,
                parent_qid=parent_entity.qid
            )
            
            new_entities.append(new_entity)
        
        return new_entities
    
    def should_continue_iteration(self, depth: int, queue_size: int) -> bool:
        """
        Determine if pipeline should continue to next iteration
        """
        if not self.config.interactive_mode:
            return True
        
        print("\n" + "="*50)
        print(f"ITERATION CHECKPOINT - Depth: {depth}")
        print(f"Queue size: {queue_size}")
        print(f"Total processed: {len(self.processed_qids)}")
        print(f"Total errors: {self.statistics.total_errors}")
        print(f"Total duplicates: {self.statistics.total_duplicates}")
        print("="*50)
        
        while True:
            response = input("\nContinue to next iteration? (y/n/p for pause): ").lower().strip()
            if response == 'y':
                return True
            elif response == 'n':
                return False
            elif response == 'p':
                self.pause()
                return False
            else:
                print("Please enter 'y' for yes, 'n' for no, or 'p' for pause")
    
    def run(self, initial_entity: WikiEntity):
        """
        Main pipeline execution
        """
        try:
            self.state = PipelineState.RUNNING
            self.statistics.start_time = datetime.now()
            
            # Initialize queue with initial entity
            if not self.queue and initial_entity.qid not in self.processed_qids:
                self.queue.append(initial_entity)
            
            logger.info(f"Starting pipeline with queue size: {len(self.queue)}")
            
            extraction_count = 0
            current_depth = 0
            
            while self.queue and self.state == PipelineState.RUNNING:
                # Process batch
                batch = []
                batch_depth = None
                
                for _ in range(min(self.config.batch_size, len(self.queue))):
                    if self.queue:
                        entity = self.queue.popleft()
                        if batch_depth is None:
                            batch_depth = entity.depth
                        
                        # Only add to batch if same depth (for iteration control)
                        if entity.depth == batch_depth:
                            batch.append(entity)
                        else:
                            # Put back for next iteration
                            self.queue.appendleft(entity)
                            break
                
                if not batch:
                    continue
                
                # Check if we should continue at this depth
                if batch_depth > current_depth:
                    current_depth = batch_depth
                    if not self.should_continue_iteration(current_depth, len(self.queue) + len(batch)):
                        # Put batch back in queue
                        for entity in reversed(batch):
                            self.queue.appendleft(entity)
                        break
                
                # Process batch with thread pool
                with ThreadPoolExecutor(max_workers=self.config.max_workers) as executor:
                    futures = {
                        executor.submit(self.process_single_entity, entity): entity 
                        for entity in batch
                    }
                    
                    for future in as_completed(futures):
                        entity = futures[future]
                        try:
                            new_entities = future.result()
                            if new_entities:
                                self.queue.extend(new_entities)
                                extraction_count += 1
                                
                                # Auto-save state periodically
                                if extraction_count % self.config.auto_save_interval == 0:
                                    self._save_state()
                                    self.update_master_excel()
                                    
                        except Exception as e:
                            logger.error(f"Failed to process entity {entity.title}: {e}")
                            traceback.print_exc()
                
                # Pause between batches
                time.sleep(self.config.pause_between_requests)
                
                # Display progress
                if extraction_count % 10 == 0:
                    self._display_progress()
            
            # Final save and update
            self._save_state()
            self.update_master_excel()
            
            if self.state == PipelineState.RUNNING:
                self.state = PipelineState.COMPLETED
                self.statistics.end_time = datetime.now()
                
            logger.info("Pipeline execution completed")
            self._display_final_statistics()
            
        except Exception as e:
            logger.error(f"Pipeline execution failed: {e}")
            traceback.print_exc()
            self.state = PipelineState.ERROR
            self._save_state()
    
    def process_single_entity(self, entity: WikiEntity) -> List[WikiEntity]:
        """
        Process a single entity and return new entities from its links
        """
        try:
            # Skip if already processed
            if entity.qid in self.processed_qids:
                logger.debug(f"Entity {entity.qid} already processed")
                return []
            
            # Extract data
            data = asyncio.run(self.extract_entity(entity))
            if not data:
                return []
            
            # Save data
            self.save_entity_data(data, entity)
            
            # Mark as processed
            with self.lock:
                self.processed_qids.add(entity.qid)
                self.statistics.total_extracted += 1
            
            # Process links for next iteration
            new_entities = self.process_entity_links(data, entity)
            
            return new_entities
            
        except Exception as e:
            logger.error(f"Error processing entity {entity.title}: {e}")
            raise
    
    def pause(self):
        """Pause pipeline execution"""
        self.state = PipelineState.PAUSED
        logger.info("Pipeline paused")
    
    def resume(self):
        """Resume pipeline execution"""
        if self.state == PipelineState.PAUSED:
            self.state = PipelineState.RUNNING
            logger.info("Pipeline resumed")
    
    def stop(self):
        """Stop pipeline execution"""
        self.state = PipelineState.STOPPED
        self._save_state()
        logger.info("Pipeline stopped")
    
    def _display_progress(self):
        """Display current progress"""
        stats = self.statistics.get_summary()
        logger.info(f"Progress - Extracted: {stats['total_extracted']}, "
                   f"Errors: {stats['total_errors']}, "
                   f"Duplicates: {stats['total_duplicates']}, "
                   f"Queue: {len(self.queue)}, "
                   f"Avg time: {stats['average_extraction_time']:.2f}s")
    
    def _display_final_statistics(self):
        """Display final statistics"""
        stats = self.statistics.get_summary()
        print("\n" + "="*60)
        print("PIPELINE EXECUTION SUMMARY")
        print("="*60)
        print(f"Total entities extracted: {stats['total_extracted']}")
        print(f"Total errors: {stats['total_errors']}")
        print(f"Total duplicates skipped: {stats['total_duplicates']}")
        print(f"Average extraction time: {stats['average_extraction_time']:.2f} seconds")
        print(f"Total runtime: {stats['total_runtime_seconds']:.2f} seconds")
        print(f"Entities by type:")
        for entity_type, qids in self.type_mapping.items():
            print(f"  - {entity_type}: {len(qids)}")
        print("="*60)
    
    def get_status(self) -> Dict:
        """Get current pipeline status"""
        return {
            'state': self.state.value,
            'queue_size': len(self.queue),
            'processed': len(self.processed_qids),
            'errors': len(self.error_qids),
            'statistics': self.statistics.get_summary(),
            'types': {t: len(qids) for t, qids in self.type_mapping.items()}
        }


# # Example usage and testing
# def extract_wikipedia_page_optimized(page_title: str) -> Dict:
#     """
#     This is where you'll implement your actual Wikipedia extraction function
#     For now, using a mock implementation
#     """
#     # TODO: Replace with actual implementation
#     import random
    
#     # Simulate API call
#     time.sleep(random.uniform(0.5, 2.0))
    
#     # Mock response structure matching your example
#     mock_data = {
#         "title": page_title,
#         "qid": f"Q{abs(hash(page_title)) % 10000000}",
#         "type": random.choice(["human", "place", "event", "organization", "concept"]),
#         "content": {
#             "extract": f"Sample extract for {page_title}...",
#             "summary": f"Summary of {page_title}"
#         },
#         "links": {
#             "internal_links": [
#                 {
#                     "title": f"Related_{i}_{page_title}",
#                     "qid": f"Q{abs(hash(f'{page_title}_{i}')) % 10000000}",
#                     "type": random.choice(["human", "place", "event", "organization"]),
#                     "shortDesc": f"Description for related item {i}",
#                     "redirectTitle": None if i % 3 else f"Redirect_to_{i}"
#                 }
#                 for i in range(random.randint(2, 5))
#             ]
#         },
#         "infobox": {
#             "field1": "value1",
#             "field2": "value2"
#         },
#         "tables": [],
#         "images": [],
#         "metadata": {
#             "page_id": str(abs(hash(page_title)) % 100000000),
#             "page_length": random.randint(1000, 50000),
#             "last_modified": datetime.now().isoformat()
#         }
#     }
    
#     return mock_data


def main():
    """
    Main entry point for the pipeline
    """
    print("="*60)
    print("WIKIPEDIA DATA EXTRACTION PIPELINE v2.0")
    print("="*60)
    
    # Configuration
    config = PipelineConfig(
        base_dir="wikipedia_data",
        max_depth=3,
        batch_size=5,
        max_workers=3,
        pause_between_requests=1.0,
        auto_save_interval=10,
        enable_deduplication=True,
        interactive_mode=True,
        retry_attempts=3
    )
    
    # Create pipeline
    pipeline = WikipediaExtractionPipeline(config)
    
    # Check for existing state
    if pipeline.processed_qids:
        print(f"\nFound existing state with {len(pipeline.processed_qids)} processed entities")
        resume = input("Resume from previous state? (y/n): ").lower().strip()
        if resume != 'y':
            # Clear state
            pipeline.processed_qids.clear()
            pipeline.queue.clear()
            pipeline.error_qids.clear()
            pipeline.type_mapping.clear()
            pipeline.statistics = PipelineStatistics()
    
    # Get initial entity if needed
    if not pipeline.queue:
        print("\nNo entities in queue. Please provide initial entity.")
        title = input("Enter Wikipedia page title: ").strip()
        if not title:
            title = "Ganga Singh"  # Default for testing
        
        initial_entity = WikiEntity(
            qid=f"Q5521008",
            title=title,
            type="human",
            depth=0
        )
    else:
        initial_entity = None
        print(f"\nResuming with {len(pipeline.queue)} entities in queue")
    
    # Run pipeline
    try:
        if initial_entity:
            pipeline.run(initial_entity)
        elif pipeline.queue:
            pipeline.state = PipelineState.RUNNING
            pipeline.run(WikiEntity(qid="", title="", type=""))  # Dummy entity, queue already has data
        
    except KeyboardInterrupt:
        print("\n\nPipeline interrupted by user")
        pipeline.pause()
        pipeline._save_state()
        print("State saved. You can resume later.")
    
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        traceback.print_exc()
    
    finally:
        # Display final status
        status = pipeline.get_status()
        print(f"\nFinal Status: {status['state']}")
        print(f"Total processed: {status['processed']}")
        print(f"Remaining in queue: {status['queue_size']}")


if __name__ == "__main__":
    main()